# Chapter 3 — Gaussian Filters: KF / EKF / UKF / IF

**Three learning tiers:**
1. **Tier 1 — Toy 1D KF**: see Gaussian conditioning with your own eyes
2. **Tier 2 — EKF**: linearization via Jacobian + finite-difference sanity check
3. **Tier 3 — UKF + IF**: sigma points vs Jacobian; information matrix view

**Kalman gain intuition:**
$$K_t = \Sigma_t^- C^\top (C \Sigma_t^- C^\top + Q)^{-1}$$
When Q is small → measurement pulls hard. When $\Sigma_t^-$ is small → prior is rigid.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse

from pluto_filters.pluto_filters.kalman_filters.ekf import EKF, LANDMARKS, normalize_angle
from pluto_filters.pluto_filters.kalman_filters.ukf import UKF
from pluto_experiments.pluto_experiments.filter_showdown.benchmark import (
    generate_figure8_trajectory, generate_measurements, rmse, nees
)

def cov_ellipse(ax, mu, sigma, n_std=2, **kw):
    """Draw a 2D covariance ellipse."""
    vals, vecs = np.linalg.eigh(sigma[:2,:2])
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
    w, h  = 2 * n_std * np.sqrt(vals)
    ell   = Ellipse(xy=mu[:2], width=w, height=h, angle=angle, **kw)
    ax.add_patch(ell)

print("Imports OK")

## Tier 1 — Toy 1D Kalman Filter: Gaussian Conditioning

In [ ]:
# ── TIER 1: 1D KF — scalar robot, noisy sensor ──────────────────────────────
np.random.seed(42)
N_STEPS   = 40
TRUE_VEL  = 0.2          # m/s
DT        = 0.5
R_noise   = 0.5          # measurement noise variance
Q_noise   = 0.01         # process noise variance

# Generate ground truth and noisy measurements
true_pos   = np.cumsum(np.ones(N_STEPS) * TRUE_VEL * DT)
measures   = true_pos + np.random.normal(0, np.sqrt(R_noise), N_STEPS)

# KF update
mu, sigma  = 0.0, 10.0   # prior: wide uncertainty
mu_hist, sigma_hist = [mu], [sigma]
K_hist     = []

for z in measures:
    # Predict
    mu    = mu + TRUE_VEL * DT
    sigma = sigma + Q_noise
    # Update (scalar: K = sigma / (sigma + R))
    K     = sigma / (sigma + R_noise)
    mu    = mu + K * (z - mu)
    sigma = (1 - K) * sigma
    K_hist.append(K)
    mu_hist.append(mu)
    sigma_hist.append(sigma)

mu_arr, sig_arr = np.array(mu_hist[1:]), np.array(sigma_hist[1:])
t_arr = np.arange(1, N_STEPS + 1) * DT

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Tier 1 — 1D Kalman Filter: Gaussian Conditioning', fontsize=13)

# (A) Estimate vs truth
ax = axes[0]
ax.fill_between(t_arr, mu_arr - 2*sig_arr, mu_arr + 2*sig_arr,
                alpha=0.25, color='steelblue', label='±2σ')
ax.plot(t_arr, true_pos,  'k-',  lw=2, label='Ground truth')
ax.plot(t_arr, measures,  'g.',  ms=6, alpha=0.7, label='Measurements')
ax.plot(t_arr, mu_arr,    'b-',  lw=2, label='KF estimate')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Position [m]')
ax.set_title('(A) KF tracks truth within ±2σ'); ax.legend(fontsize=8)

# (B) Kalman gain convergence
ax = axes[1]
ax.plot(t_arr, K_hist, 'purple', lw=2)
ax.axhline(np.sqrt(Q_noise) / (np.sqrt(Q_noise) + np.sqrt(R_noise)),
           color='red', ls='--', label='Steady-state K')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Kalman Gain K')
ax.set_title('(B) K converges to steady-state\nK≈1: trust measurement; K≈0: trust prior')
ax.legend(fontsize=8)

# (C) Uncertainty (σ) convergence
ax = axes[2]
ax.plot(t_arr, sig_arr, 'teal', lw=2)
ax.axhline(Q_noise + R_noise * Q_noise / (Q_noise + R_noise),
           color='red', ls='--', label='Steady-state σ')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Uncertainty σ')
ax.set_title('(C) σ converges from wide prior to small value')
ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig('ch03_tier1_kf.png', dpi=120); plt.show()
print(f"Initial σ={10.0:.1f} → Final σ={sig_arr[-1]:.4f}")
print(f"Steady-state Kalman gain: {K_hist[-1]:.4f}")

## Tier 2 — EKF with Jacobian Finite-Difference Check

In [ ]:
# ── TIER 2: EKF Jacobian finite-difference check ────────────────────────────
def measurement_model(pose, lx, ly):
    """h(x): range and bearing to landmark."""
    dx  = lx - pose[0]; dy = ly - pose[1]
    r   = np.sqrt(dx**2 + dy**2)
    phi = normalize_angle(np.arctan2(dy, dx) - pose[2])
    return np.array([r, phi])

def jacobian_analytic(pose, lx, ly):
    """H = dh/dx (analytic)."""
    dx = lx - pose[0]; dy = ly - pose[1]
    r2 = dx**2 + dy**2; r  = np.sqrt(r2)
    return np.array([
        [-dx/r,      -dy/r,       0],
        [ dy/r2,     -dx/r2,     -1]
    ])

def jacobian_finitediff(pose, lx, ly, eps=1e-5):
    """H = dh/dx (finite difference)."""
    H = np.zeros((2, 3))
    for i in range(3):
        dp = pose.copy(); dp[i] += eps
        dm = pose.copy(); dm[i] -= eps
        H[:, i] = (measurement_model(dp, lx, ly) - measurement_model(dm, lx, ly)) / (2*eps)
    return H

# Test at 5 random poses
print("Jacobian check: analytic vs finite-difference (max abs error per landmark)\n")
test_poses = [np.array([1.0, 0.5, 0.3]),
              np.array([2.0, -1.0, 1.2]),
              np.array([-1.0, 2.0, -0.7])]

for pose in test_poses:
    for lm_id, (lx, ly) in LANDMARKS.items():
        H_a = jacobian_analytic(pose, lx, ly)
        H_fd = jacobian_finitediff(pose, lx, ly)
        err = np.abs(H_a - H_fd).max()
        status = '✅' if err < 1e-5 else '❌'
        print(f"{status} pose=({pose[0]:.1f},{pose[1]:.1f},{pose[2]:.1f})  "
              f"LM{lm_id}  max|H_analytic - H_fd| = {err:.2e}")
print("\n→ If all ✅, your Jacobian is correct.")

In [ ]:
# ── EKF vs UKF on figure-8 ───────────────────────────────────────────────
np.random.seed(42)
true_poses, dt = generate_figure8_trajectory(300, 0.1)
measurements   = generate_measurements(true_poses, landmark_id=0)

def run_filter(filter_cls, v=0.3, omega=0.2, return_sigmas=False):
    mu0    = np.array([0.3, 0.2, 0.15])
    sigma0 = np.diag([1.0, 1.0, 0.2])
    f      = filter_cls(mu0, sigma0)
    ests, covs = [], []
    for i, (r, phi) in enumerate(measurements):
        if i > 0: f.predict(v, omega, dt)
        f.update(0, r, phi)
        ests.append(f.mu.copy())
        covs.append(f.sigma.copy())
    return np.array(ests), covs

ekf_ests, ekf_covs = run_filter(EKF)
ukf_ests, ukf_covs = run_filter(UKF)

ekf_rmse = rmse(ekf_ests, true_poses)
ukf_rmse = rmse(ukf_ests, true_poses)
ekf_nees = nees(ekf_ests, ekf_covs, true_poses)
ukf_nees = nees(ukf_ests, ukf_covs, true_poses)

print(f"EKF: RMSE={ekf_rmse:.4f}m  NEES={ekf_nees:.3f}  (consistent if NEES ≈ n_state=3)")
print(f"UKF: RMSE={ukf_rmse:.4f}m  NEES={ukf_nees:.3f}")

In [ ]:
# ── Visualize trajectories + covariance ellipses ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, ests, covs, name, color in [
    (axes[0], ekf_ests, ekf_covs, 'EKF', 'steelblue'),
    (axes[1], ukf_ests, ukf_covs, 'UKF', 'coral'),
]:
    ax.plot(true_poses[:,0], true_poses[:,1], 'k-', lw=2, label='Ground Truth', zorder=5)
    ax.plot(ests[:,0], ests[:,1], '--', color=color, lw=1.5, label=f'{name} estimate')
    for i in range(0, len(ests), 20):
        cov_ellipse(ax, ests[i], covs[i], n_std=2,
                    fill=False, edgecolor=color, alpha=0.4, lw=1)
    for lm_id, (lx, ly) in LANDMARKS.items():
        ax.plot(lx, ly, 'k^', ms=9)
        ax.annotate(f'LM{lm_id}', (lx+0.1, ly+0.1), fontsize=7)
    ax.set_title(f'{name}  RMSE={rmse(ests, true_poses):.4f}m  NEES={nees(ests, covs, true_poses):.2f}')
    ax.legend(fontsize=8); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.suptitle('EKF vs UKF — figure-8 with covariance ellipses', fontsize=13)
plt.tight_layout(); plt.savefig('ch03_gaussian_filters.png', dpi=120); plt.show()

## Tier 3 — Information Filter (IF)

In [ ]:
# ── TIER 3: Information Filter — information matrix grows with each measurement
# IF state: Omega (information matrix) = Sigma^{-1}, xi (info vector) = Omega @ mu
print("Information Filter demo — 1D case")

mu_if   = 0.0
Omega   = 1e-4          # start with almost no information (wide prior)
xi      = Omega * mu_if

IF_mus, IF_omegas = [], []
measures_1d = measures  # reuse from Tier 1

for z in measures_1d:
    # Predict (process noise Q adds to covariance → subtracts from information)
    Sigma = 1.0 / Omega
    Sigma = Sigma + Q_noise
    Omega = 1.0 / Sigma
    xi    = Omega * (xi / (Omega + 1e-10))  # approximate info predict

    # Update (add information from measurement)
    H = 1.0          # scalar observation model
    R_inv = 1.0 / R_noise
    Omega = Omega + H**2 * R_inv
    xi    = xi + H * R_inv * z

    IF_mus.append(xi / Omega)
    IF_omegas.append(Omega)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Tier 3 — Information Filter: Ω grows with measurements', fontsize=13)

ax = axes[0]
ax.plot(t_arr, IF_mus, 'teal', lw=2, label='IF estimate')
ax.plot(t_arr, true_pos, 'k-', lw=2, label='Ground truth')
ax.fill_between(t_arr,
                np.array(IF_mus) - 2/np.sqrt(np.array(IF_omegas)),
                np.array(IF_mus) + 2/np.sqrt(np.array(IF_omegas)),
                alpha=0.2, color='teal', label='±2σ')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Position [m]')
ax.set_title('IF estimate vs truth'); ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_arr, IF_omegas, 'teal', lw=2)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Information Ω = Σ⁻¹')
ax.set_title('Ω grows with each measurement\n(high Ω = confident; 2 independent sensors → Ω₁+Ω₂)')
plt.tight_layout(); plt.savefig('ch03_tier3_if.png', dpi=120); plt.show()

## ✅ Compare | 🔥 Break | 📏 Measure

In [ ]:
# COMPARE: innovation (measurement residual) over time
print("=== COMPARE: EKF vs UKF RMSE and NEES ===")
print(f"EKF  RMSE={ekf_rmse:.4f}m   NEES={ekf_nees:.3f}")
print(f"UKF  RMSE={ukf_rmse:.4f}m   NEES={ukf_nees:.3f}")
print(f"\nNEES should be ≈ 3 (state dimension) for a consistent filter.")
print(f"NEES >> 3 → filter is overconfident (true uncertainty > reported)")
print(f"NEES << 3 → filter is underconfident\n")

# BREAK: remove landmarks one by one
print("=== BREAK: degrade measurement availability ===")
for n_lm in [5, 3, 1]:
    available = {k: v for i, (k, v) in enumerate(LANDMARKS.items()) if i < n_lm}
    meas_limited = generate_measurements(true_poses, landmark_id=0)
    ekf_lim = EKF(np.array([0.3, 0.2, 0.15]), np.diag([1.0, 1.0, 0.2]))
    ests_lim = []
    for i, (r, phi) in enumerate(meas_limited):
        if i > 0: ekf_lim.predict(0.3, 0.2, 0.1)
        ekf_lim.update(0, r, phi)
        ests_lim.append(ekf_lim.mu.copy())
    print(f"  {n_lm} landmark(s): EKF RMSE={rmse(np.array(ests_lim), true_poses):.4f}m")

# MEASURE
print("\n=== MEASURE: RMSE and NEES summary ===")
print(f"{'Filter':>6}  {'RMSE [m]':>10}  {'NEES':>8}  {'Consistent?':>12}")
print("-" * 45)
for name, r_, n_ in [('EKF', ekf_rmse, ekf_nees), ('UKF', ukf_rmse, ukf_nees)]:
    consistent = 'YES' if abs(n_ - 3.0) < 3.0 else 'NO (overconf)' if n_ > 3 else 'NO (underconf)'
    print(f"{name:>6}  {r_:>10.4f}  {n_:>8.3f}  {consistent:>12}")